# 10. Validation Summary

Aggregates synthetic validation results to answer:
1. Does GS correctly identify BE vs SC animals?
2. Does SBI correctly identify BE vs SC animals?
3. Do GS and SBI agree?
4. How does accuracy vary across scenarios and fit targets?


In [ ]:
%matplotlib inline
from shared_setup import *
import pickle
from pathlib import Path


In [ ]:
# ── Mode ──────────────────────────────────────────────────────────────────
# 'load' = load cluster results from results/
# 'run'  = quick local execution (small settings)
MODE = 'load'

if MODE == 'run':
    N_SYNTHETIC = 3
    N_GS_SEEDS = 2
    BURN_IN = 500
    N_SBI_SIMS = 1_000
    N_CV_REPEATS = 4
    SEED = 42
    print('MODE: run (quick local)')
else:
    print('MODE: load (cluster results)')


## 1. Load / Generate Validation Results

In [ ]:
RESULTS_DIR = Path('../results/validation')

if MODE == 'load':
    # ── Load from cluster ──────────────────────────────────────────────
    synth_gs_dfs = {}
    synth_sbi_dfs = {}

    for cohort in ['static_uniform', 'learning_uniform']:
        for ft in ['update_matrix', 'conditional_psych']:
            # GS
            gs_path = RESULTS_DIR / 'synth_gs' / f'{cohort}_{ft}' / 'summary.pkl'
            if gs_path.exists():
                with open(gs_path, 'rb') as f:
                    synth_gs_dfs[(cohort, ft)] = pickle.load(f)['df']
                print(f'Loaded GS: {cohort}/{ft}')

            # SBI
            sbi_dir = RESULTS_DIR / 'synth_sbi' / f'{cohort}_{ft}'
            if sbi_dir.exists():
                sbi_files = sorted(sbi_dir.glob('synth_*.pkl'))
                if sbi_files:
                    rows = []
                    for p in sbi_files:
                        with open(p, 'rb') as f:
                            d = pickle.load(f)
                        rows.append(d)
                    synth_sbi_dfs[(cohort, ft)] = pd.DataFrame(rows)
                    print(f'Loaded SBI: {cohort}/{ft}')

elif MODE == 'run':
    # ── Quick local run ────────────────────────────────────────────────
    from analysis.validation import (
        make_synthetic_cohort, make_learning_cohort,
        run_gs_model_id, run_sbi_model_id, summarise_agreement,
    )

    SBI_OK = False
    try:
        import torch; SBI_OK = True
    except ImportError:
        pass

    synth_gs_dfs = {}
    synth_sbi_dfs = {}

    # Static uniform
    static = make_synthetic_cohort(n_per_model=N_SYNTHETIC, burn_in=BURN_IN, seed=SEED)
    for ft in ['update_matrix', 'conditional_psych']:
        gs = run_gs_model_id(static, n_seeds=N_GS_SEEDS, burn_in=BURN_IN, fit_target=ft)
        synth_gs_dfs[('static_uniform', ft)] = gs

    # Learning uniform
    learning = make_learning_cohort(n_per_model=N_SYNTHETIC, burn_in=BURN_IN, seed=SEED)
    for a in learning:
        a['expert_sessions'] = a['sessions'][-8:]
    for ft in ['update_matrix', 'conditional_psych']:
        gs = run_gs_model_id(learning, sessions_key='expert_sessions',
                             n_seeds=N_GS_SEEDS, burn_in=BURN_IN, fit_target=ft)
        synth_gs_dfs[('learning_uniform', ft)] = gs

    # SBI (if available)
    if SBI_OK:
        for ft in ['update_matrix', 'conditional_psych']:
            sbi = run_sbi_model_id(static, n_sbi_sims=N_SBI_SIMS,
                                   n_cv_repeats=N_CV_REPEATS, burn_in=BURN_IN,
                                   seed=SEED, method=ft)
            synth_sbi_dfs[('static_uniform', ft)] = sbi


## 2. Accuracy Summary

In [ ]:
summary_rows = []

for (cohort, ft), gs_df in synth_gs_dfs.items():
    if len(gs_df) == 0:
        continue
    row = {'scenario': cohort, 'fit_target': ft}

    if 'correct' in gs_df.columns:
        row['gs_accuracy'] = gs_df['correct'].mean()
        row['gs_n'] = len(gs_df)
    elif 'gs_correct' in gs_df.columns:
        row['gs_accuracy'] = gs_df['gs_correct'].mean()
        row['gs_n'] = len(gs_df)

    sbi_key = (cohort, ft)
    if sbi_key in synth_sbi_dfs:
        sbi_df = synth_sbi_dfs[sbi_key]
        if 'correct' in sbi_df.columns:
            row['sbi_accuracy'] = sbi_df['correct'].mean()
            row['sbi_n'] = len(sbi_df)
        elif 'sbi_correct' in sbi_df.columns:
            row['sbi_accuracy'] = sbi_df['sbi_correct'].mean()
            row['sbi_n'] = len(sbi_df)

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
if len(summary) > 0:
    print(summary.to_string(index=False, float_format='%.2f'))
else:
    print('No validation results found.')


## 3. Accuracy by Scenario and Fit Target

In [ ]:
if len(summary) > 0 and 'gs_accuracy' in summary.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── GS accuracy ────────────────────────────────────────────────────
    ax = axes[0]
    for i, ft in enumerate(['update_matrix', 'conditional_psych']):
        sub = summary[summary['fit_target'] == ft]
        if len(sub) == 0:
            continue
        x = np.arange(len(sub))
        offset = -0.2 + 0.4 * i
        bars = ax.bar(x + offset, sub['gs_accuracy'], 0.35,
                      label=ft, alpha=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels(sub['scenario'], rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('GS Accuracy')
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='grey', ls='--', lw=0.8)
    ax.legend(fontsize=8)
    ax.set_title('Grid Search: Model ID Accuracy')

    # ── SBI accuracy ───────────────────────────────────────────────────
    ax = axes[1]
    if 'sbi_accuracy' in summary.columns:
        for i, ft in enumerate(['update_matrix', 'conditional_psych']):
            sub = summary[(summary['fit_target'] == ft) & summary['sbi_accuracy'].notna()]
            if len(sub) == 0:
                continue
            x = np.arange(len(sub))
            offset = -0.2 + 0.4 * i
            ax.bar(x + offset, sub['sbi_accuracy'], 0.35,
                   label=ft, alpha=0.7)
            ax.set_xticks(x)
            ax.set_xticklabels(sub['scenario'], rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('SBI Accuracy')
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color='grey', ls='--', lw=0.8)
    ax.legend(fontsize=8)
    ax.set_title('SBI: Model ID Accuracy')

    plt.tight_layout()
    plt.show()
else:
    print('Not enough data to plot.')


## 4. Interpretation

**If GS accuracy is high (>90%) for both fit targets**: methods are
reliable. Proceed with confidence to real data.

**If UM and conditional_psych diverge**: one target is more discriminative.
Use both on real data and compare.

**If SBI accuracy is lower than GS**: SBI has the known SC-bias issue.
Use GS as primary, SBI as supplementary.

**If learning_uniform accuracy drops vs static_uniform**: expert-phase
model ID is affected by learning history. Interpret 3a results cautiously.
